# all_tables.ipynb

The purpose of this notebook is to investigate whether all tables (with the exception of c_a) may be naively joined to hd. The notebook outputs a .csv file for each table naively joined to hd.

In [ ]:
import duckdb
from pathlib import Path
import pandas as pd

current_script_parent = Path.cwd().parent
con = duckdb.connect(current_script_parent / "ipeds.duckdb")

all_tables = ["ic", "ic_ay", "ic_py", "adm", "efia", "effy", 
              "ef_a", "ef_b", "ef_c", "ef_d", "sfa", "gr", "gr200", 
              "om", "eap", "sal_is", "al", "f1a", "f2", "f3"]

query = """
SELECT 
    h.unitid,
    h.year,
    h.institution_name,
    h.city,
    h.state,
    h.carnegie,  
    i.* EXCLUDE (unitid, year)
FROM hd h
LEFT JOIN {table} i 
    ON h.unitid = i.unitid 
   AND h.year = i.year
WHERE h.year = 2023;
"""

# get and print shape of hd table
hd_df = con.execute("SELECT * FROM hd WHERE year = 2023").fetchdf()
print(f"HD table shape: {hd_df.shape}")

for table in all_tables:
    this_query = query.replace("{table}", table)

    print(f"Processing table: {table}")

    df = con.execute(this_query).fetchdf()

    print(f"Saving table: {table} with shape {df.shape}")

    df.to_csv(current_script_parent / "experiments" / f"ipeds_{table}_2023.csv", index=False)


HD table shape: (6163, 208)
Processing table: ic
Saving table: ic with shape (6163, 254)
Processing table: ic_ay
Saving table: ic_ay with shape (6163, 298)
Processing table: ic_py
Saving table: ic_py with shape (6163, 138)
Processing table: adm
Saving table: adm with shape (6163, 114)
Processing table: efia
Saving table: efia with shape (6163, 23)
Processing table: effy
Saving table: effy with shape (116635, 155)
Processing table: ef_a
Saving table: ef_a with shape (115406, 160)
Processing table: ef_b
Saving table: ef_b with shape (159167, 28)
Processing table: ef_c
Saving table: ef_c with shape (44366, 12)
Processing table: ef_d
Saving table: ef_d with shape (6163, 51)
Processing table: sfa
Saving table: sfa with shape (6163, 716)
Processing table: gr
Saving table: gr with shape (53805, 149)
Processing table: gr200
Saving table: gr200 with shape (6163, 58)
Processing table: om
Saving table: om with shape (49879, 72)
Processing table: eap
Saving table: eap with shape (288431, 44)
Proce

So it seems that ic, ic_ay, ic_py, adm, efia, ef_d, sfa, gr200, al, f1a, f2, f3 may be naively joined.

effy, ef_a, ef_b, ef_c, gr, om, eap, sal_is will reqiure some pivoting.

## effy
12-month unduplicated headcount by race/ethnicity, gender and level of student: 2023-24

Seems like you would pivot on `EFFYALEV`: Level and degree/certificate-seeking status of student

Going to leave this one out for now as I don't think headcount by race would be super important.

## ef_a
Race/ethnicity, gender, attendance status, and level of student.

Seems like you might have to pivot on `EFALEVEL`, `LINE`, `SECTION`, and `LSTUDY`?

Not sure how this is different from effy. Skipping.

## ef_b
Age category, gender, attendance status, and level of student.

Seems like you'd have to pivot on `efbage` and `line`.

Skipping demographic information.

## ef_c
Residence and migration of first-time freshman.

Seems like you want to pivot on `efcstate`.

The below script gets a pivoted ef_c. But lots of schools are missing data (like Union College & Columbia University)




In [ ]:
# ef_c

query = """
SELECT 
    h.unitid,
    h.year,
    h.institution_name,
    h.city,
    h.state AS institution_state,
    h.carnegie,  
    ef_pivoted.* EXCLUDE (unitid)
FROM (
    SELECT * FROM hd WHERE year = 2023
) h
LEFT JOIN (
    PIVOT (
        SELECT unitid, efcstate, efres01 
        FROM ef_c 
        WHERE year = 2023
    )
    ON efcstate
    USING SUM(efres01)
    GROUP BY unitid
) ef_pivoted
    ON h.unitid = ef_pivoted.unitid;
"""

print(f"ef_c table shape: {hd_df.shape}")

# Execute the query
df_migration = con.execute(query).fetchdf()

# drop rows with 65 or more NaNs (no data)
df_migration = df_migration[df_migration.isna().sum(axis=1) <= 64]

df_migration = df_migration.fillna(0)

df_migration.to_csv(current_script_parent / "experiments" / "data" / "ipeds_ef_c_2023.csv", index=False)

ef_c table shape: (6163, 71)
